# Integrated cost-plus-schedule risk deliverable

This notebook is the product's one report path (SPEC 1.2 §6a, R-4). It runs the engines in
memory, gates the result on cross-validation, composes the report, writes the Word and Stage-3
standalone artefacts and the state cache, and renders itself to HTML through nbconvert.

**It names no project.** Every input path comes from the environment and every methodology
value from the selected project's register; the heading below is the binding's. Run it through
`RUN_FINAL.bat`, which sets nothing it was not given and refuses on anything unset.

**On a cross-validation FAIL** every artefact is still written, carrying the FAIL banner, and
the last cell then raises so the run exits non-zero (D-11: write, then fail). Files with a
banner are evidence; an aborted run that writes nothing is not.

In [ ]:
# Cell 1 - run the deliverable: pre-flight, engines, gate, compose, write.
# Everything numeric happens in src/ (the pure-data rule); this notebook renders it.
from __future__ import annotations

import os
import sys
import warnings
from pathlib import Path

from IPython.display import HTML, Markdown, display

warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")

REPO_ROOT = Path(os.getcwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.final.deliverable import run_deliverable  # noqa: E402

# The single pre-flight refuses here, once, naming every missing setting (A-1).
run = run_deliverable()
composed = run.composed
cv = run.cross_validation

In [ ]:
# Cell 2 - title, product and run identity, all from the binding and the run (never a literal).
# The project is shown by its public name, never its binding key (HTMLPROV-01).
from src.final.run_identity import front_page_markdown, run_identity_markdown  # noqa: E402

display(Markdown(front_page_markdown(run.inputs)))
display(Markdown(run_identity_markdown(run.inputs, run.provenance)))

## Cross-validation

In [ ]:
# Cell 3 - the gate. Banner first when it failed, so a reader of the HTML meets it before
# any number. The run does NOT stop here - D-11 writes everything, then fails at the end.
if not cv["all_pass"]:
    display(HTML(
        "<div style='border:3px solid #C00000; padding:14px; text-align:center; "
        "font-family:Arial,sans-serif; font-weight:bold; color:#C00000; font-size:16pt; "
        "margin-bottom:14px;'>"
        "CROSS-VALIDATION FAILED.<br/>THIS REPORT MUST NOT BE DISTRIBUTED.<br/>"
        "INVESTIGATE BEFORE PROCEEDING.</div>"
    ))

from src.final.presentation_contract import VALUE_LABELS


def _mark(ok):
    return "PASS" if ok else "**FAIL**"

display(Markdown("\n".join([
    "| Check | Result | Detail |",
    "|---|---|---|",
    f"| Cost marginal | {_mark(cv['cost_passes_calibration'])} | "
    f"gap ${cv['cost_absolute_gap_dollars']:,.0f} ({cv['cost_relative_gap_pct']*100:.3f}%) |",
    f"| Schedule marginal | {_mark(cv['schedule_passes_calibration'])} | "
    f"gap {cv['schedule_absolute_gap_days']:.3f} days |",
    f"| Top-3 cost drivers | {_mark(cv['top_3_cost_contributors_consistent'])} | "
    f"cost-only {', '.join(cv['top_3_cost_contributors_cost_only'])}; "
    f"integrated {', '.join(cv['top_3_cost_contributors_integrated'])} |",
    f"| Top-3 schedule drivers | {_mark(cv['top_3_schedule_contributors_consistent'])} | "
    f"standalone {', '.join(cv['top_3_schedule_contributors_sr5'])}; "
    f"integrated {', '.join(cv['top_3_schedule_contributors_integrated'])} |",
    f"| **Overall** | **{_mark(cv['all_pass'])}** | basis: {VALUE_LABELS.get(cv['schedule_top3_basis'], cv['schedule_top3_basis'])} |",
])))

## Executive summary

In [ ]:
# Cell 4 - one renderer. What each composed value shows as - table, labelled row, prose,
# chart - and each page's order are decided in src/final/html_blocks.py from the presentation
# contract (DELIVFIX-01: dicts came out as one-line paragraphs, and keys as code names). This
# cell only displays the blocks. Anything unrecognised is still shown, never dropped: a page
# that quietly vanishes from a client deliverable is worse than one that renders badly.
from IPython.display import Image

from src.final.html_blocks import blocks


def _as_path(value):
    """The value as an existing filesystem path, or None. Never raises on odd input."""
    if not isinstance(value, (str, Path)):
        return None
    try:
        p = Path(value)
        return p if p.exists() and p.is_file() else None
    except (TypeError, ValueError, OSError):
        return None


def text(value):
    """A Markdown object that always renders the STRING, never a file it happens to name.

    ``IPython.display.Markdown("some/existing/path")`` does not render that text: the
    DisplayObject constructor notices the string names an existing file and reads the file
    instead. For this deliverable that turns a chart path into a UnicodeDecodeError on the
    PNG's first bytes, and would silently inline a file's contents where it did not. Passing
    the text as ``data=`` keyword with ``filename=None`` is not enough - the check is on the
    value - so anything that resolves to a file is rendered as a literal path.
    """
    if _as_path(value) is not None:
        return Markdown(f"`{value}`")
    return Markdown(str(value))


def render(value, *, section=None):
    """Display one composed value. WHAT to show, and how, is src/final/html_blocks.py's."""
    for block in blocks(value, section=section):
        kind, payload = block[0], block[1]
        if kind == "image":
            display(Image(filename=payload))
        elif kind == "table":
            display(HTML(payload.to_html(index=block[2])))
        else:
            display(text(payload))


def show(key, *, heading=None):
    value = composed.get(key)
    if value is None:
        return
    if heading:
        display(Markdown(heading))
    render(value, section=key)


show("page1_headline")
show("page1_headline_interpretation")
show("page1_range_table")
show("callouts")

In [ ]:
# Cell 5 - the pages, in order, each rendered from the composed dict. The order is
# the presentation contract's (src/final/presentation_contract.py, HTML_SECTIONS);
# tests/test_presentation_contract.py reds naming the section if the two diverge.
PAGES = [
    ("page2_distribution", "## Distribution"),
    ("page3_cost_drivers", "## Cost drivers"),
    ("page4_schedule_drivers", "## Schedule drivers"),
    ("page5_reconciliation", "## Reconciliation"),
    ("page6_action_implications", "## Actions and implications"),
    ("page7_methodology", "## Methodology and caveats"),
    ("page8_audit_trail", "## Audit trail"),
]
missing = []
for key, heading in PAGES:
    section = composed.get(key)
    if section is None:
        missing.append(key)
        continue
    display(Markdown(heading))
    render(section, section=key)

# A page that vanishes because a composer key was renamed must not vanish quietly: the
# reader of a client deliverable cannot tell an absent section from one that was never
# meant to be there. Absences are reported, on the page, by name.
if missing:
    display(Markdown(
        "> **Sections not produced by this run:** " + ", ".join(f"`{k}`" for k in missing)
        + ". Expected for a run whose composition omits them; if a section you expect is "
          "listed here, the composer key has moved."
    ))

## Appendices

In [ ]:
# Cell 6 - appendices and the disclaimer.
show("appendix_counterfactual", heading="### Mode-classification sensitivity")
show("appendix_methodology", heading="### Methodology")
show("appendix_audit_trail", heading="### Audit trail")
show("disclaimer_paragraphs", heading="### Disclaimer")

## Artefacts written by this run

In [ ]:
# Cell 7 - what this run produced, and the provenance that identifies it.
# Basenames and the digest method in words; the full paths stay in provenance.json (HTMLPROV-01).
from src.final.run_identity import artefacts_markdown, provenance_markdown  # noqa: E402

display(Markdown(artefacts_markdown(run.artefacts)))
display(Markdown(provenance_markdown(run.provenance)))

In [ ]:
# Cell 8 - D-11's second half, and it must stay LAST.
#
# Every artefact above exists on disk by now and carries the FAIL banner where the gate
# failed. Only now does the run fail, so nbconvert exits non-zero with the evidence written
# rather than aborting mid-sequence and leaving a stack trace and no files.
#
# Do not move this cell, and do not add a cell after it: anything below would not execute on
# a failing run, which is how the previous notebook's comment and code came to disagree.
run.raise_if_failed()
display(Markdown("**Cross-validation passed.** All artefacts above are clear for review."))